# Tokenization

Tokenization is at the heart
- Why can't LLM spell words? Tokenization.
- Why can't LLM do super simple string processing tasks like reversing a string? Tokenization.
- Why is LLM worse at non-English langyuages? Tokenization.
- Why is LLM bad at simple arithemetic? Tokenization.
- Why did GPT-2 have more than necessary troiuble in cobding in Python? Tokenization.
- Why did my LLM abruptly halt when it sees the string "<|endofftext>"? Tokenization.
- What is this weird warning get about a "trailing whitespace"? Tokenization.
- Why the LLM break if I ask it about "SolidGoldMagikarp"? Tokenization.
- Why should I prefer to use YAML over JSON with LLMs? Tokenization.
- Why is LLM not actually end-to-end language modelling? Tokenization.
- What is the real root of suffering? Tokenization.


---

Good tokenization web app: [https://tiktokenizer.vercel.app](https://tiktokenizer.vercel.app)

Example string:

```
Tokenization is at the heart of much weirdness of LLMs. Do not brush it off.

127 + 677 = 804
1275 + 6773 = 8041

Egg.
I have an Egg.
egg.
EGG.

만나서 반가워요. 저는 OpenAI에서 개발한 대규모 언어 모델인 ChatGPT입니다. 궁금한 것이 있으시면 무엇이든 물어보세요.

for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
```

---

Much glory awaits someone who can delete the need for tokenization. But meanwhile, let's learn about it.

In [1]:
ord('🎀')

127872

In [2]:
[ord(x) for x in 'happy birthday 🎀']

[104, 97, 112, 112, 121, 32, 98, 105, 114, 116, 104, 100, 97, 121, 32, 127872]

In [3]:
ord(' ')

32

In [4]:
'happy birthday 🎀'.encode("utf-8")

b'happy birthday \xf0\x9f\x8e\x80'

In [5]:
print(list('happy birthday 🎀'.encode("utf-8")))
len(list('happy birthday 🎀'.encode("utf-8")))

[104, 97, 112, 112, 121, 32, 98, 105, 114, 116, 104, 100, 97, 121, 32, 240, 159, 142, 128]


19

In [6]:
len(list('happy birthday 🎀'.encode("utf-32")))

68

In [7]:
len(list('happy birthday 🎀'.encode("utf-16")))

36

The history of why we are sticking to utf-8 is quiet interesting do check it out
and as we can see 
```python
        list('happy birthday 🎀'.encode("utf-8"))
```
gives the number to represent every atomic element there is in text.
But that will be doing as like we did in bigram language model and it will bloat out everything there is while training Transformer. So, so many bad news.

So, we define the BPE

In [8]:
len('happy birthday 🎀')

16

In [107]:
text = """🎀The history of why we are sticking to utf-8 is quiet interesting do check it out
and as we can see 
```python
        list('happy birthday 🎀'.encode("utf-8"))
```
gives the number to represent every atomic element there is in text.
But that will be doing as like we did in bigram language model and it will bloat out everything there is while training Transformer. So, so many bad news.

So, we define the BPE"""

tokens = text.encode("utf-8")

tokens = list(map(int, tokens))
print(len(tokens))
print(len(text))

416
410


In [108]:
# understanding map
def x2(x):
    return x**2
x = [2,3]
# y = map(x2,x)
y = list(map(lambda x:x**2,x))
y

[4, 9]

In [109]:
l = [1,3,4,5]
l[1:]

[3, 4, 5]

In [110]:
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair,0) +1
    return counts
stats = get_stats(tokens)
# print(stats)
print(sorted(((v,k)for k, v in stats.items()), reverse = True))

[(16, (101, 32)), (10, (105, 110)), (9, (116, 32)), (9, (32, 116)), (8, (116, 104)), (8, (32, 119)), (8, (32, 105)), (7, (101, 114)), (7, (32, 32)), (6, (121, 32)), (6, (115, 32)), (6, (114, 101)), (6, (110, 103)), (6, (104, 101)), (6, (97, 110)), (5, (117, 116)), (5, (105, 115)), (5, (103, 32)), (5, (32, 98)), (5, (32, 97)), (4, (119, 101)), (4, (116, 111)), (4, (115, 116)), (4, (111, 32)), (4, (100, 32)), (4, (96, 96)), (4, (32, 100)), (3, (118, 101)), (3, (114, 121)), (3, (114, 97)), (3, (110, 116)), (3, (110, 32)), (3, (108, 32)), (3, (105, 108)), (3, (104, 105)), (3, (101, 115)), (3, (101, 110)), (3, (100, 101)), (3, (97, 116)), (3, (32, 115)), (3, (32, 111)), (3, (32, 108)), (3, (32, 101)), (2, (240, 159)), (2, (159, 142)), (2, (142, 128)), (2, (121, 116)), (2, (119, 105)), (2, (119, 104)), (2, (116, 105)), (2, (116, 102)), (2, (116, 101)), (2, (115, 101)), (2, (112, 121)), (2, (111, 117)), (2, (111, 114)), (2, (111, 100)), (2, (111, 44)), (2, (110, 101)), (2, (110, 100)), (2, (1

## Understanding rough stuff

In [111]:
# me trying to understand how this above thing works
x = {1:"hi",
     2:"bi"}
x.get(2,0)

'bi'

In [112]:
zip()

In [113]:
l = [1,2,3]
m = [6,7]
list(zip(l,m))

[(1, 6), (2, 7)]

# Back to tokenization babyy

In [114]:
# pick the element x from stats so that stats.get(x) is as large as possible
top_pair = max(stats, key = stats.get)
top_pair

(101, 32)

In [115]:
help(max)

Help on built-in function max in module builtins:

max(...)
    max(iterable, *[, default=obj, key=func]) -> value
    max(arg1, arg2, *args, *[, key=func]) -> value

    With a single iterable argument, return its biggest item. The
    default keyword-only argument specifies an object to return if
    the provided iterable is empty.
    With two or more arguments, return the largest argument.



## Training

In [116]:
def merge(ids, top_pair, idx):
    new_ids =[]
    i = 0
    while i < len(ids):
        if i<len(ids)-1 and ids[i] == top_pair[0] and ids[i+1] == top_pair[1] :
            new_ids.append(idx)
            i+=2
        else:
            new_ids.append(ids[i])
            i+=1
            # d[idx] = d.get(idx,0)+1
            
    return new_ids

new_stats = merge([5,4,6,7], (4,6), 99)
new_stats
# sorted(new_stats, key = new_stats.get, reverse= True)
# print(sorted(((v,k)for k, v in new_stats.items()), reverse = True))


[5, 99, 7]

In [117]:
def merge(ids, top_pair, idx):
    new_ids =[]
    i = 0
    while i < len(ids):
        if i<len(ids)-1 and ids[i] == top_pair[0] and ids[i+1] == top_pair[1] :
            new_ids.append(idx)
            i+=2
        else:
            new_ids.append(ids[i])
            i+=1
            # d[idx] = d.get(idx,0)+1
            
    return new_ids

tokens2 = merge(tokens, top_pair, 256)

print(len(tokens2), len(tokens))
# sorted(new_stats, key = new_stats.get, reverse= True)
# print(sorted(((v,k)for k, v in new_stats.items()), reverse = True))


400 416


In [126]:
text = """🎀The history of why we are sticking to utf-8 is quite interesting do check it out and as we can see ```python list('happy birthday 🎀'.encode("utf-8")) ``` gives the number to represent every atomic element there is in text but that will be doing as like we did in bigram language model and it will bloat out everything there is while training Transformer so so many bad news so we define the BPE minbpe minimal clean code for the byte-level Byte Pair Encoding (BPE) algorithm commonly used in LLM tokenization this algorithm was popularized for LLMs by the GPT-2 paper and the associated GPT-2 code release from OpenAI Sennrich et al 2015 is cited as the original reference for the use of BPE in NLP applications today all modern LLMs (e.g. GPT Llama Mistral) use this algorithm to train their tokenizers there are two Tokenizers in this repository both of which can perform the 3 primary functions of a Tokenizer 1) train the tokenizer vocabulary and merges on a given text 2) encode from text to tokens 3) decode from tokens to text the files of the repo are as follows minbpe/base.py implements the Tokenizer class which is the base class it contains the train encode and decode stubs save/load functionality and there are also a few common utility functions this class is not meant to be used directly but rather to be inherited from minbpe/basic.py implements the BasicTokenizer the simplest implementation of the BPE algorithm that runs directly on text minbpe/regex.py implements the RegexTokenizer that further splits the input text by a regex pattern which is a preprocessing stage that splits up the input text by categories (think letters numbers punctuation) before tokenization this ensures that no merges will happen across category boundaries this was introduced in the GPT-2 paper and continues to be in use as of GPT-4 this class also handles special tokens if any minbpe/gpt4.py implements the GPT4Tokenizer this class is a light wrapper around the RegexTokenizer (2 above) that exactly reproduces the tokenization of GPT-4 in the tiktoken library the wrapping handles some details around recovering the exact merges in the tokenizer and the handling of some unfortunate (and likely historical) 1-byte token permutations finally the script train.py trains the two major tokenizers on the input text tests/taylorswift.txt (this is the Wikipedia entry for her kek) and saves the vocab to disk for visualization this script runs in about 25 seconds on my (M1) MacBook all of the files above are very short and thoroughly commented and also contain a usage example on the bottom of the file quick start as the simplest example we can reproduce the Wikipedia article on BPE as follows from minbpe import BasicTokenizer tokenizer = BasicTokenizer() text = "aaabdaaabac" tokenizer.train(text, 256 + 3) # 256 are the byte tokens then do 3 merges print(tokenizer.encode(text)) # [258, 100, 258, 97, 99] print(tokenizer.decode([258, 100, 258, 97, 99])) # aaabdaaabac tokenizer.save("toy") # writes two files: toy.model (for loading) and toy.vocab (for viewing) according to Wikipedia running bpe on the input string: "aaabdaaabac" for 3 merges results in the string: "XdXac" where X=ZY Y=ab and Z=aa the tricky thing to note is that minbpe always allocates the 256 individual bytes as tokens and then merges bytes as needed from there so for us a=97 b=98 c=99 d=100 (their ASCII values) then when (a,a) is merged to Z Z will become 256 likewise Y will become 257 and X 258 so we start with the 256 bytes and do 3 merges to get to the result above with the expected output of [258, 100, 258, 97, 99] inference GPT-4 comparison we can verify that the RegexTokenizer has feature parity with the GPT-4 tokenizer from tiktoken as follows text = "hello123!!!? (안녕하세요!) 😉" # tiktoken import tiktoken enc = tiktoken.get_encoding("cl100k_base") print(enc.encode(text)) # [15339, 4513, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 57037] # ours from minbpe import GPT4Tokenizer tokenizer = GPT4Tokenizer() print(tokenizer.encode(text)) # [15339, 4513, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 57037] (you'll have to pip install tiktoken to run) under the hood the GPT4Tokenizer is just a light wrapper around RegexTokenizer passing in the merges and the special tokens of GPT-4 we can also ensure the special tokens are handled correctly text = "<|endoftext|>hello world" # tiktoken import tiktoken enc = tiktoken.get_encoding("cl100k_base") print(enc.encode(text, allowed_special="all")) # [100257, 15339, 1917] # ours from minbpe import GPT4Tokenizer tokenizer = GPT4Tokenizer() print(tokenizer.encode(text, allowed_special="all")) # [100257, 15339, 1917] note that just like tiktoken we have to explicitly declare our intent to use and parse special tokens in the call to encode otherwise this can become a major footgun unintentionally tokenizing attacker-controlled data (e.g. user prompts) with special tokens the allowed_special parameter can be set to "all" "none" or a list of special tokens to allow training unlike tiktoken this code allows you to train your own tokenizer in principle and to my knowledge if you train the RegexTokenizer on a large dataset with a vocabulary size of 100K you would reproduce the GPT-4 tokenizer there are two paths you can follow first you can decide that you don't want the complexity of splitting and preprocessing text with regex patterns and you also don't care for special tokens in that case reach for the BasicTokenizer you can train it and then encode and decode for example as follows from minbpe import BasicTokenizer tokenizer = BasicTokenizer() tokenizer.train(very_long_training_string, vocab_size=4096) tokenizer.encode("hello world") # string -> tokens tokenizer.decode([1000, 2000, 3000]) # tokens -> string tokenizer.save("mymodel") # writes mymodel.model and mymodel.vocab tokenizer.load("mymodel.model") # loads the model back the vocab is just for vis if you instead want to follow along with OpenAI did for their text tokenizer it's a good idea to adopt their approach of using regex pattern to split the text by categories the GPT-4 pattern is a default with the RegexTokenizer so you'd simple do something like from minbpe import RegexTokenizer tokenizer = RegexTokenizer() tokenizer.train(very_long_training_string, vocab_size=32768) tokenizer.encode("hello world") # string -> tokens tokenizer.decode([1000, 2000, 3000]) # tokens -> string tokenizer.save("tok32k") # writes tok32k.model and tok32k.vocab tokenizer.load("tok32k.model") # loads the model back from disk where of course you'd want to change around the vocabulary size depending on the size of your dataset special tokens finally you might wish to add special tokens to your tokenizer register these using the register_special_tokens function for example if you train with vocab_size of 32768 then the first 256 tokens are raw byte tokens the next 32768-256 are merge tokens and after those you can add the special tokens the last "real" merge token will have id of 32767 (vocab_size - 1) so your first special token should come right after that with an id of exactly 32768 so: from minbpe import RegexTokenizer tokenizer = RegexTokenizer() tokenizer.train(very_long_training_string, vocab_size=32768) tokenizer.register_special_tokens({"<|endoftext|>": 32768}) tokenizer.encode("<|endoftext|>hello world", allowed_special="all") you can of course add more tokens after that as you like finally I'd like to stress that I tried hard to keep the code itself clean readable and hackable you should not have feel scared to read the code and understand how it works the tests are also a nice place to look for more usage examples that reminds me: tests we use the pytest library for tests all of them are located in the tests/ directory first pip install pytest if you haven't already then: $ pytest -v . to run the tests (-v is verbose slightly prettier) community extensions gnp/minbpe-rs: A Rust implementation of minbpe providing (near) one-to-one correspondence with the Python version exercise for those trying to study BPE here is the advised progression exercise for how you can build your own minbpe step by step see exercise.md lecture I built the code in this repository in this YouTube video you can also find this lecture in text form in lecture.md todos write a more optimized Python version that could run over large files and big vocabs write an even more optimized C or Rust version (think through) rename GPT4Tokenizer to GPTTokenizer and support GPT-2/GPT-3/GPT-3.5 as well? write a LlamaTokenizer similar to GPT4Tokenizer (i.e. attempt sentencepiece equivalent) license MIT all this"""

tokens = text.encode("utf-8")

tokens = list(map(int, tokens))
print(len(tokens))
print(len(text))

8668
8649


In [127]:
#BPE Algorith
def get_stats(ids:list):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0)+1
    return counts

def merge(ids, top_pair, idx):
    new_ids =[]
    i = 0
    while i < len(ids):
        if i<len(ids)-1 and ids[i] == top_pair[0] and ids[i+1] == top_pair[1] :
            new_ids.append(idx)
            i+=2
        else:
            new_ids.append(ids[i])
            i+=1            
    return new_ids

vocab_size = 276
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
total_merges = 0
for i in range(num_merges):
    stats = get_stats(ids)
    
    top_pair = max(stats, key = stats.get)
    idx = 256+i
    print(f"merging {top_pair}  which had {stats[top_pair]} occurances into {idx}")
    total_merges += stats[top_pair]
    ids =merge(ids, top_pair, idx)
    merges[top_pair] = idx
print(len(tokens))
print(len(ids))
print("total number of merges are", total_merges)
print(f"compression ratio: {len(tokens)/ len(ids):.2f}")
# merges
# tokens

merging (32, 116)  which had 287 occurances into 256
merging (101, 32)  which had 205 occurances into 257
merging (101, 110)  which had 165 occurances into 258
merging (256, 104)  which had 135 occurances into 259
merging (101, 114)  which had 134 occurances into 260
merging (105, 110)  which had 127 occurances into 261
merging (115, 32)  which had 113 occurances into 262
merging (111, 107)  which had 111 occurances into 263
merging (263, 258)  which had 105 occurances into 264
merging (116, 32)  which had 84 occurances into 265
merging (105, 122)  which had 79 occurances into 266
merging (259, 257)  which had 71 occurances into 267
merging (32, 97)  which had 67 occurances into 268
merging (264, 266)  which had 67 occurances into 269
merging (111, 114)  which had 65 occurances into 270
merging (269, 260)  which had 63 occurances into 271
merging (111, 32)  which had 55 occurances into 272
merging (100, 32)  which had 55 occurances into 273
merging (101, 120)  which had 54 occurances i

In [128]:
8664-6572

2092

In [129]:
merges

{(32, 116): 256,
 (101, 32): 257,
 (101, 110): 258,
 (256, 104): 259,
 (101, 114): 260,
 (105, 110): 261,
 (115, 32): 262,
 (111, 107): 263,
 (263, 258): 264,
 (116, 32): 265,
 (105, 122): 266,
 (259, 257): 267,
 (32, 97): 268,
 (264, 266): 269,
 (111, 114): 270,
 (269, 260): 271,
 (111, 32): 272,
 (100, 32): 273,
 (101, 120): 274,
 (97, 108): 275}

## Decoding
Given a sequence of integers in range [0, vocab_size], what is the text?

In [157]:
def decode(ids):
    return bytes(ids).decode('utf-8')

decode(tokens)

'🎀The history of why we are sticking to utf-8 is quite interesting do check it out and as we can see ```python list(\'happy birthday 🎀\'.encode("utf-8")) ``` gives the number to represent every atomic element there is in text but that will be doing as like we did in bigram language model and it will bloat out everything there is while training Transformer so so many bad news so we define the BPE minbpe minimal clean code for the byte-level Byte Pair Encoding (BPE) algorithm commonly used in LLM tokenization this algorithm was popularized for LLMs by the GPT-2 paper and the associated GPT-2 code release from OpenAI Sennrich et al 2015 is cited as the original reference for the use of BPE in NLP applications today all modern LLMs (e.g. GPT Llama Mistral) use this algorithm to train their tokenizers there are two Tokenizers in this repository both of which can perform the 3 primary functions of a Tokenizer 1) train the tokenizer vocabulary and merges on a given text 2) encode from text to

In [151]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0]+vocab[p1]
print(vocab)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [142]:
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text= tokens.decode('utf-8')
    return text
decode([128])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

In [152]:
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text= tokens.decode('utf-8', errors = 'replace')
    return text
decode([128])

'�'

In [153]:
decode(ids)

'🎀The history of why we are sticking to utf-8 is quite interesting do check it out and as we can see ```python list(\'happy birthday 🎀\'.encode("utf-8")) ``` gives the number to represent every atomic element there is in text but that will be doing as like we did in bigram language model and it will bloat out everything there is while training Transformer so so many bad news so we define the BPE minbpe minimal clean code for the byte-level Byte Pair Encoding (BPE) algorithm commonly used in LLM tokenization this algorithm was popularized for LLMs by the GPT-2 paper and the associated GPT-2 code release from OpenAI Sennrich et al 2015 is cited as the original reference for the use of BPE in NLP applications today all modern LLMs (e.g. GPT Llama Mistral) use this algorithm to train their tokenizers there are two Tokenizers in this repository both of which can perform the 3 primary functions of a Tokenizer 1) train the tokenizer vocabulary and merges on a given text 2) encode from text to

## let's Encode
Other way around: given string what are the tokens

In [178]:
print(vocab)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [183]:
stoi = {v:k for k,v in vocab.items()}
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >=2:
        stats= get_stats(tokens)
        pair = min(stats, key = lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)

    return tokens

encode('he;lkpo')

[104, 101, 59, 108, 107, 112, 111]

In [177]:
stats

{(240, 159): 3,
 (159, 142): 2,
 (142, 128): 2,
 (128, 84): 1,
 (84, 104): 1,
 (104, 257): 1,
 (257, 104): 3,
 (104, 105): 7,
 (105, 115): 24,
 (115, 116): 36,
 (116, 270): 5,
 (270, 121): 5,
 (121, 32): 41,
 (32, 111): 21,
 (111, 102): 28,
 (102, 32): 24,
 (32, 119): 23,
 (119, 104): 8,
 (104, 121): 1,
 (119, 257): 10,
 (257, 97): 24,
 (97, 114): 32,
 (114, 257): 14,
 (257, 115): 8,
 (116, 105): 20,
 (105, 99): 21,
 (99, 107): 8,
 (107, 261): 1,
 (261, 103): 40,
 (103, 256): 7,
 (256, 272): 29,
 (272, 117): 2,
 (117, 116): 9,
 (116, 102): 2,
 (102, 45): 2,
 (45, 56): 2,
 (56, 32): 4,
 (32, 105): 20,
 (105, 262): 28,
 (262, 113): 1,
 (113, 117): 3,
 (117, 105): 5,
 (105, 116): 38,
 (116, 257): 7,
 (257, 261): 5,
 (261, 116): 10,
 (116, 260): 13,
 (260, 101): 1,
 (101, 115): 35,
 (116, 261): 3,
 (103, 32): 17,
 (32, 100): 12,
 (100, 272): 4,
 (272, 99): 3,
 (99, 104): 8,
 (104, 101): 7,
 (101, 99): 40,
 (107, 32): 6,
 (105, 265): 5,
 (265, 111): 4,
 (111, 117): 47,
 (117, 265): 6,
 (265